# 06 — OCR + сборка submission CSV + оценка по GT

**Что делает:**
1. Берёт ~500 best-кропов из `data/best_crops/<video>/<tid>.jpg`
2. Прогоняет PaddleOCR (ru) + WeChatQRCode на каждом
3. Через `parse_all` из `05_parsers.ipynb` извлекает поля
4. Пишет `data/submission.csv` со всеми 23 полями
5. Матчит треки

**Вход:** `data/best_crops/`, `data/best_crops_meta.json`, `data/per_frame_tracks.json`, `data/crops_meta.json`  
**Выход:** `data/submission.csv`, `data/eval_report.json`

## 1. Импорты и загрузка парсеров из 05_parsers.ipynb

In [ ]:
import json
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import nbformat

ROOT = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
BEST_CROPS_DIR  = ROOT / 'data' / 'best_crops'
BEST_META       = ROOT / 'data' / 'best_crops_meta.json'
PER_FRAME       = ROOT / 'data' / 'per_frame_tracks.json'
GT_META         = ROOT / 'data' / 'crops_meta.json'
SUBMISSION_CSV  = ROOT / 'data' / 'submission.csv'
EVAL_REPORT     = ROOT / 'data' / 'eval_report.json'

# Загружаем все определения из 05_parsers.ipynb
parsers_nb = nbformat.read(ROOT / 'eda' / '05_parsers.ipynb', as_version=4)
for cell in parsers_nb.cells:
    if cell.cell_type == 'code':
        exec(cell.source, globals())

print('Парсеры загружены из 05_parsers.ipynb')
print('Доступны:', [n for n in ('parse_all', 'parse_qr_payload', 'OcrLine') if n in globals()])

## 2. Инициализация PaddleOCR + WeChat QR

In [3]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang='ru', use_textline_orientation=True)
print('PaddleOCR готов')

try:
    qr_reader = cv2.wechat_qrcode_WeChatQRCode()
    print('WeChat QR готов')
except Exception as e:
    qr_reader = None
    print(f'WeChat QR недоступен: {e}')

/Users/alex/ML Projects/Lenta Tech Life Hack/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/alex/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/alex/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/alex/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating mo

PaddleOCR готов
WeChat QR готов


In [4]:
import os
import json
import time
import cv2

from collections import defaultdict

VIDEO_MAP = {
    '25_12-20': ROOT / 'Данные/25_12-20/25_12-20.mp4',
    '26_12-20': ROOT / 'Данные/26_12-20/26_12-20.mp4',
    '43_15': ROOT / 'Данные/43_15/43_15.mp4',
    'Unlabeled_25_12-20': ROOT / 'Данные/Unlabeled/25_12-20.mp4',
    'Unlabeled_26_12-20': ROOT / 'Данные/Unlabeled/26_12-20.mp4',
    'Unlabeled_26_2-10': ROOT / 'Данные/Unlabeled/26_2-10.mp4',
}

PAD_RATIO = 0.0

meta_all = json.loads(BEST_META.read_text(encoding='utf-8'))

by_video = defaultdict(list)

for m in meta_all:
    by_video[m['video']].append(m)

for vid in by_video:
    by_video[vid].sort(key=lambda r: r['best_frame_idx'])

t0 = time.time()
total = 0

for vid, tracks in by_video.items():
    cap = cv2.VideoCapture(str(VIDEO_MAP[vid]))

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    for m in tracks:
        cap.set(cv2.CAP_PROP_POS_FRAMES, m['best_frame_idx'])

        ok, frame = cap.read()

        if not ok:
            continue

        x1, y1, x2, y2 = m['bbox']

        w = x2 - x1
        h = y2 - y1

        px = w * PAD_RATIO
        py = h * PAD_RATIO

        x1p = max(0, int(x1 - px))
        y1p = max(0, int(y1 - py))
        x2p = min(W, int(x2 + px))
        y2p = min(H, int(y2 + py))

        crop = frame[y1p:y2p, x1p:x2p]

        if crop.size == 0:
            continue

        cv2.imwrite(
            str(ROOT / m['crop_path']),
            crop,
            [cv2.IMWRITE_JPEG_QUALITY, 95],
        )

        total += 1

    cap.release()

    print(f'  {vid}: {len(tracks)} треков')

print(f'\nГотово: {total} кропов за {(time.time() - t0) / 60:.1f} мин')

sample = meta_all[0]
sz = os.path.getsize(ROOT / sample['crop_path'])
img = cv2.imread(str(ROOT / sample['crop_path']))

print(f'Пример: {sample["crop_path"]}  {img.shape}  ({sz // 1024} KB)')

  25_12-20: 109 треков
  26_12-20: 153 треков
  43_15: 41 треков
  Unlabeled_25_12-20: 86 треков
  Unlabeled_26_12-20: 88 треков
  Unlabeled_26_2-10: 67 треков

Готово: 542 кропов за 1.9 мин
Пример: data/best_crops/25_12-20/0001.jpg  (278, 272, 3)  (47 KB)


## 3. Helper: один кроп → OcrLine[] + QR payload

In [5]:
def ocr_lines(img: np.ndarray) -> list:
    """PaddleOCR на BGR-изображении → список OcrLine."""
    if img is None or img.size == 0:
        return []
    res = ocr.predict(img)
    if not res or not res[0]:
        return []
    out = []

    r = res[0]
    if isinstance(r, dict):
        texts = r.get('rec_texts', [])
        scores = r.get('rec_scores', [])
        boxes = r.get('rec_boxes', r.get('rec_polys', []))
        for t, s, b in zip(texts, scores, boxes):
            arr = np.array(b).reshape(-1, 2)
            x = float(arr[:, 0].min()); y = float(arr[:, 1].min())
            w = float(arr[:, 0].max() - x); h = float(arr[:, 1].max() - y)
            out.append(OcrLine(t, x, y, w, h, float(s)))
    else:
        
        for item in r:
            box, (text, score) = item[0], item[1]
            arr = np.array(box)
            x = float(arr[:, 0].min()); y = float(arr[:, 1].min())
            w = float(arr[:, 0].max() - x); h = float(arr[:, 1].max() - y)
            out.append(OcrLine(text, x, y, w, h, float(score)))
    return out


def decode_qr(img: np.ndarray) -> dict:
    """WeChat QR декодер. Возвращает dict с QR-полями или пустой dict."""
    if qr_reader is None or img is None or img.size == 0:
        return {}
    try:
        texts, _ = qr_reader.detectAndDecode(img)
    except Exception:
        return {}
    for t in texts or []:
        if t:
            parsed = parse_qr_payload(t)
            if parsed:
                return parsed
    return {}


import json as _json
_meta = json.loads(BEST_META.read_text())
_sample = _meta[0]
_img = cv2.imread(str(ROOT / _sample['crop_path']))
_lines = ocr_lines(_img)
_qr = decode_qr(_img)
print(f"Кроп: {_sample['crop_path']}")
print(f"  OCR строк: {len(_lines)}")
for ln in _lines[:5]:
    print(f"    {ln.text!r}  conf={ln.conf:.2f}")
print(f"  QR полей: {len(_qr)} → {_qr}")
print(f"\n  parse_all → {parse_all(_lines)}")

Кроп: data/best_crops/25_12-20/0001.jpg
  OCR строк: 12
    'Напиток'  conf=0.78
    'безалкогальный'  conf=0.87
    'ABRAU LGHT'  conf=0.95
    'Apomar. c raa.'  conf=0.82
    '(РОCСHя) 0. 25L'  conf=0.66
  QR полей: 0 → {}

  parse_all → {'barcode': '4680140271438', 'id_sku': '770207716035', 'print_datetime': None, 'discount_amount': '-10%', 'additional_info': None, 'price_default': None, 'price_card': '129,99'}


In [6]:
import zxingcpp

OCR_CACHE = ROOT / 'data' / 'ocr_cache.json'

def decode_barcode_1d(img):
    """zxing-cpp на BGR-кропе → строка 1D-штрихкода или None."""
    if img is None or img.size == 0:
        return None
    try:
        results = zxingcpp.read_barcodes(img)
    except Exception:
        return None
    for r in results:
        fmt = str(r.format).split('.')[-1]
        if fmt in ('EAN13', 'EAN8', 'UPCA', 'UPCE', 'Code128', 'Code39', 'ITF'):
            return r.text
    return None


def line_to_dict(L):
    return {'text': L.text, 'x': L.x, 'y': L.y, 'w': L.w, 'h': L.h, 'conf': L.conf}

def dict_to_line(d):
    return OcrLine(d['text'], d['x'], d['y'], d['w'], d['h'], d['conf'])


def build_product_name(lines):
    """Склеиваем alpha-линии из верхней половины кропа в правильном порядке по y.
    Это даёт что-то типа 'Напиток безалкогальный ABRAU LGHT Apomar. c raa. (РОCСHя) 0. 25L'."""
    if not lines: return None
    max_y = max(L.y + L.h for L in lines)
    cutoff = max_y * 0.55   # верхние 55% (название всегда вверху ценника)
    top = [L for L in lines if (L.y + L.h / 2) < cutoff
                            and any(c.isalpha() for c in L.text)
                            and len(L.text.strip()) >= 2]
    if not top: return None
    # Сортируем сверху вниз
    top.sort(key=lambda L: (L.y, L.x))
    name = ' '.join(L.text.strip() for L in top)
    return name if len(name) >= 5 else None


ocr_cache = {}
if OCR_CACHE.exists():
    ocr_cache = json.loads(OCR_CACHE.read_text())
    print(f'Загружен OCR-кэш: {len(ocr_cache)} кропов')
else:
    print('OCR-кэш не найден, будет создан при первом прогоне')


CSV_FIELDS = [
    'video', 'track_id',
    'product_name', 'price_default', 'price_card', 'price_discount',
    'barcode', 'discount_amount', 'id_sku', 'print_datetime',
    'code', 'additional_info', 'color', 'special_symbols',
    'qr_code_barcode', 'price1_qr', 'price2_qr', 'price3_qr', 'price4_qr',
    'wholesale_level_1_count', 'wholesale_level_1_price',
    'wholesale_level_2_count', 'wholesale_level_2_price',
    'action_price_qr', 'action_code_qr',
]

LIMIT_FIRST_N = None
FORCE_REOCR  = False

meta = json.loads(BEST_META.read_text())
if LIMIT_FIRST_N: meta = meta[:LIMIT_FIRST_N]

records = []
n_zxing_hit = n_qr_hit = n_cache_hit = 0
t0 = time.time()
for i, m in enumerate(meta, 1):
    img_path = ROOT / m['crop_path']
    cache_key = m['crop_path']

    if not FORCE_REOCR and cache_key in ocr_cache:
        c = ocr_cache[cache_key]
        lines = [dict_to_line(d) for d in c['lines']]
        qr = c['qr']
        zxing_barcode = c['zxing']
        n_cache_hit += 1
    else:
        img = cv2.imread(str(img_path))
        lines = ocr_lines(img)
        qr = decode_qr(img)
        zxing_barcode = decode_barcode_1d(img)
        ocr_cache[cache_key] = {
            'lines': [line_to_dict(L) for L in lines],
            'qr': qr,
            'zxing': zxing_barcode,
        }

    if zxing_barcode: n_zxing_hit += 1
    if qr: n_qr_hit += 1

    parsed = parse_all(lines) if lines else {}

    rec = {f: 'нет' for f in CSV_FIELDS}
    rec['video']    = m['video']
    rec['track_id'] = m['track_id']
    rec['color']    = 'red'
    rec['price_discount']  = 'нет'
    rec['code']            = 'нет'
    rec['special_symbols'] = 'нет'

    for k in ('barcode', 'id_sku', 'print_datetime', 'discount_amount',
              'additional_info', 'price_card', 'price_default'):
        if parsed.get(k):
            rec[k] = parsed[k]

    if zxing_barcode:
        rec['barcode'] = zxing_barcode

    for k, v in qr.items():
        if k in rec and v is not None:
            rec[k] = v

    pn = build_product_name(lines)
    if pn:
        rec['product_name'] = pn

    records.append(rec)
    if i % 50 == 0:
        rate = i / (time.time() - t0)
        eta = (len(meta) - i) / rate
        print(f'  {i}/{len(meta)}  ({rate:.1f} crops/s, ETA {eta/60:.1f} мин, '
              f'cache={n_cache_hit}, zxing={n_zxing_hit}, qr={n_qr_hit})')

OCR_CACHE.write_text(json.dumps(ocr_cache, ensure_ascii=False))
print(f'\nГотово за {(time.time()-t0)/60:.1f} мин, записей: {len(records)}')
print(f'  cache hits: {n_cache_hit}/{len(records)} ({n_cache_hit/len(records):.0%})')
print(f'  zxing:      {n_zxing_hit}/{len(records)}')
print(f'  QR:         {n_qr_hit}/{len(records)}')

Загружен OCR-кэш: 544 кропов
  50/544  (1359.0 crops/s, ETA 0.0 мин, cache=50, zxing=0, qr=0)
  100/544  (2347.0 crops/s, ETA 0.0 мин, cache=100, zxing=0, qr=0)
  150/544  (1443.5 crops/s, ETA 0.0 мин, cache=150, zxing=0, qr=2)
  200/544  (1287.2 crops/s, ETA 0.0 мин, cache=200, zxing=0, qr=2)
  250/544  (1131.4 crops/s, ETA 0.0 мин, cache=250, zxing=0, qr=2)
  300/544  (1072.0 crops/s, ETA 0.0 мин, cache=300, zxing=0, qr=2)
  350/544  (1049.5 crops/s, ETA 0.0 мин, cache=350, zxing=0, qr=2)
  400/544  (1022.8 crops/s, ETA 0.0 мин, cache=400, zxing=0, qr=2)
  450/544  (1079.3 crops/s, ETA 0.0 мин, cache=450, zxing=0, qr=2)
  500/544  (1122.8 crops/s, ETA 0.0 мин, cache=500, zxing=0, qr=2)

Готово за 0.0 мин, записей: 544
  cache hits: 544/544 (100%)
  zxing:      0/544
  QR:         2/544


## 5. Submission CSV

In [7]:
df = pd.DataFrame(records, columns=CSV_FIELDS)
df.to_csv(SUBMISSION_CSV, sep=';', index=False, encoding='utf-8')
print(f'Записан {SUBMISSION_CSV}  ({SUBMISSION_CSV.stat().st_size//1024} KB, {len(df)} строк)')
df.head(3).T

Записан /Users/alex/ML Projects/Lenta Tech Life Hack/data/submission.csv  (105 KB, 544 строк)


,0,1,2
video,25_12-20,25_12-20,25_12-20
track_id,1,2,3
product_name,Напиток безалкогальный ABRAU LGHT Apomar. c ra...,Hаниток безалкогольный SANTO STEFANO Biаnco (P...,Вино безалогольное LE PETIT OERET Sauvignon(@p...
price_default,нет,"252,00","1308,00"
price_card,"129,99","129,99","699,99"
price_discount,нет,нет,нет
barcode,4680140271438,1670025471658,нет
discount_amount,-10%,-48%,-48%
id_sku,770207716035,2702734529,нет
print_datetime,нет,нет,нет


## 6. Оценка

In [8]:
def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0: return 0.0
    return inter / ((ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter)

gt = json.loads(GT_META.read_text())
per_frame = json.loads(PER_FRAME.read_text())
fps_map = {'25_12-20': 19.6, '26_12-20': 20.0, '43_15': 19.9}
labeled_videos = set(fps_map)

pred_lookup = {(r['video'], r['track_id']): r for r in records}

matches = []  # list of (gt_record, predicted_record)
for g in gt:
    vid = g['video']
    if vid not in labeled_videos: continue
    gt_frame = int(round(g['frame_timestamp_ms'] / 1000.0 * fps_map[vid]))
    frame_map = per_frame.get(vid, {})
    best_iou, best_tid = 0.0, None
    for delta in range(-2, 3):
        for t in frame_map.get(str(gt_frame + delta), []):
            i = iou(g['bbox'], t['bbox'])
            if i > best_iou:
                best_iou, best_tid = i, t['track_id']
    if best_iou >= 0.5 and (vid, best_tid) in pred_lookup:
        matches.append((g, pred_lookup[(vid, best_tid)]))

print(f'Совпало: {len(matches)} GT ↔ predicted')

Совпало: 124 GT ↔ predicted


In [11]:
PER_TAG_THRESHOLD = 0.80
EVAL_FIELDS = [f for f in CSV_FIELDS if f not in ('video', 'track_id')]
BASE_CORRECT_IF_MATCHED = 6
TOTAL_FIELDS_PER_TAG = len(EVAL_FIELDS) + BASE_CORRECT_IF_MATCHED

def norm(v):
    if v is None: return 'нет'
    s = str(v).strip()
    return s if s else 'нет'


pred_by_match = {id(g): p for g, p in matches}

tag_scores = []
field_stats = {f: {'correct': 0, 'total': 0, 'examples_wrong': []} for f in EVAL_FIELDS}

for g in gt:
    vid = g['video']
    if vid not in labeled_videos: continue
    matched = id(g) in pred_by_match
    if matched:
        p = pred_by_match[id(g)]
        n_correct = BASE_CORRECT_IF_MATCHED   # bbox+filename+timestamp
        for f in EVAL_FIELDS:
            gt_v = norm(g.get(f, 'нет'))
            pr_v = norm(p.get(f, 'нет'))
            field_stats[f]['total'] += 1
            if gt_v == pr_v:
                n_correct += 1
                field_stats[f]['correct'] += 1
            elif len(field_stats[f]['examples_wrong']) < 3:
                field_stats[f]['examples_wrong'].append({'gt': gt_v, 'pred': pr_v})
    else:
        n_correct = 0
    tag_scores.append({'video': vid, 'correct': n_correct,
                       'pct': n_correct / TOTAL_FIELDS_PER_TAG, 'matched': matched})


passed = sum(1 for t in tag_scores if t['pct'] >= PER_TAG_THRESHOLD)
total  = len(tag_scores)
print(f'  Ценников с точностью ≥ {PER_TAG_THRESHOLD:.0%}: {passed}/{total} = {passed/total:.1%}')

# Распределение
print('\nРаспределение per-tag accuracy (всего 157):')
buckets = [(0.0, 0.01), (0.01, 0.5), (0.5, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]
for lo, hi in buckets:
    n = sum(1 for t in tag_scores if lo <= t['pct'] < hi)
    bar = '█' * int(40 * n / total)
    pct_label = '0% (миссы)' if lo == 0 else f'{lo:.0%}-{hi:.0%}'
    print(f'  {pct_label:>13}: {n:3d} {bar}')

# Per-video
print('\nПо видео:')
for vid in sorted(labeled_videos):
    vs = [t for t in tag_scores if t['video'] == vid]
    pas = sum(1 for t in vs if t['pct'] >= PER_TAG_THRESHOLD)
    matched = sum(1 for t in vs if t['matched'])
    print(f'  {vid}: passed {pas}/{len(vs)} ({pas/len(vs):.0%}), capture {matched}/{len(vs)} ({matched/len(vs):.0%})')

# Per-field (для дебага)
print(f'\nPer-field accuracy на {len(matches)} matched (для отладки):')
rows = sorted(((f, s['correct'], s['total']) for f, s in field_stats.items()),
              key=lambda r: r[1]/r[2] if r[2] else 0, reverse=True)
for f, c, t in rows:
    print(f'  {f:<26} {c:>3}/{t:<3} = {c/t:.1%}' if t else f'  {f:<26}  (нет матчей)')

EVAL_REPORT.write_text(json.dumps({
    'main_metric_pct': passed / total,
    'passed_tags':  passed,
    'total_tags':   total,
    'matched_tags': sum(1 for t in tag_scores if t['matched']),
    'capture_rate': sum(1 for t in tag_scores if t['matched']) / total,
    'per_video':    {vid: {
        'passed':  sum(1 for t in tag_scores if t['video']==vid and t['pct']>=PER_TAG_THRESHOLD),
        'matched': sum(1 for t in tag_scores if t['video']==vid and t['matched']),
        'total':   sum(1 for t in tag_scores if t['video']==vid)} for vid in labeled_videos},
    'per_field':    field_stats,
}, ensure_ascii=False, indent=1))
print(f'\nReport saved: {EVAL_REPORT}')

  Ценников с точностью ≥ 80%: 0/157 = 0.0%

Распределение per-tag accuracy (всего 157):
     0% (миссы):  33 ████████
         1%-50%:   0 
        50%-70%: 115 █████████████████████████████
        70%-80%:   9 ██
        80%-90%:   0 
       90%-101%:   0 

По видео:
  25_12-20: passed 0/57 (0%), capture 54/57 (95%)
  26_12-20: passed 0/71 (0%), capture 42/71 (59%)
  43_15: passed 0/29 (0%), capture 28/29 (97%)

Per-field accuracy на 124 matched (для отладки):
  price_discount             124/124 = 100.0%
  price3_qr                  124/124 = 100.0%
  action_price_qr            124/124 = 100.0%
  action_code_qr             124/124 = 100.0%
  color                      123/124 = 99.2%
  wholesale_level_1_count    123/124 = 99.2%
  wholesale_level_1_price    123/124 = 99.2%
  wholesale_level_2_count    123/124 = 99.2%
  wholesale_level_2_price    123/124 = 99.2%
  price_card                  77/124 = 62.1%
  discount_amount             73/124 = 58.9%
  additional_info             62/1

In [12]:
import re as _re

# ── Plausibility-чекеры (выглядит ли значение разумно) ──
def _plausible_barcode(v):
    if v in (None, 'нет'): return None  # пропуск — нет значения
    digits = _re.sub(r'\D', '', str(v))
    if len(digits) == 13:
        return ean13_checksum_ok(digits)
    return len(digits) in (8, 12)

def _plausible_id_sku(v):
    if v in (None, 'нет'): return None
    digits = _re.sub(r'\D', '', str(v))
    return 9 <= len(digits) <= 12

def _plausible_price(v):
    if v in (None, 'нет'): return None
    s = str(v).replace(',', '.')
    try:
        f = float(s)
        return 10.0 <= f <= 99999.0
    except Exception:
        return False

def _plausible_price_card_99(v):
    return _plausible_price(v) and str(v).endswith(',99')

def _plausible_discount(v):
    if v in (None, 'нет'): return None
    m = _re.fullmatch(r'-\d{1,2}%', str(v))
    if not m: return False
    n = int(str(v)[1:-1])
    return 1 <= n <= 99

def _plausible_datetime(v):
    if v in (None, 'нет'): return None
    return bool(_re.fullmatch(r'\d{2}\.\d{2}\.\d{4}( \d{1,2}:\d{2})?', str(v)))

def _plausible_color(v):
    return v in ('red', 'yellow') if v not in (None, 'нет') else None

def _plausible_additional(v):
    return v in ADDITIONAL_HINTS if v not in (None, 'нет') else None

def _plausible_special_sym(v):
    return v in ('К', 'Ш') if v not in (None, 'нет') else None

PLAUSIBLE = {
    'barcode': _plausible_barcode,
    'id_sku': _plausible_id_sku,
    'price_default': _plausible_price,
    'price_card': _plausible_price_card_99,
    'price_discount': _plausible_price,
    'discount_amount': _plausible_discount,
    'print_datetime': _plausible_datetime,
    'color': _plausible_color,
    'additional_info': _plausible_additional,
    'special_symbols': _plausible_special_sym,
    'qr_code_barcode': _plausible_barcode,
}

# ── Soft similarity (char-level Jaccard) ──
def char_jaccard(a, b):
    """|set(a) ∩ set(b)| / |set(a) ∪ set(b)| на нормализованных строках.
    Для пары ('нет','нет') возвращает 1.0; для одной 'нет' — 0.0."""
    a, b = norm(a), norm(b)
    if a == b: return 1.0
    sa, sb = set(a.lower()), set(b.lower())
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

# ── Считаем все три метрики ──
NON_EMPTY = lambda v: v not in (None, '', 'нет')

# Coverage: процент записей с НЕ-`нет` значением
coverage = {}
for f in EVAL_FIELDS:
    n_filled = sum(1 for r in records if NON_EMPTY(r.get(f)))
    coverage[f] = n_filled / len(records)

# Plausibility: на значениях НЕ-`нет` — % прошедших чек
plausibility = {}
for f in EVAL_FIELDS:
    if f not in PLAUSIBLE: continue
    checks = [PLAUSIBLE[f](r.get(f)) for r in records]
    checks = [c for c in checks if c is not None]
    plausibility[f] = sum(checks) / len(checks) if checks else None

# Soft similarity vs GT на 124 matched
soft_sim = {}
for f in EVAL_FIELDS:
    sims = [char_jaccard(g.get(f), p.get(f)) for g, p in matches]
    soft_sim[f] = sum(sims) / len(sims) if sims else 0.0

# Strict acc уже посчитан в field_stats
def strict_acc(f):
    s = field_stats.get(f)
    return s['correct'] / s['total'] if s and s['total'] else 0.0

# ── Вывод таблицы ──
print(f'{"FIELD":<26} {"COVERAGE":>10} {"PLAUSIBLE":>12} {"SOFT SIM":>10} {"STRICT":>9}')
print('─' * 70)
for f in EVAL_FIELDS:
    cov = f'{coverage[f]:.0%}'
    plaus = f'{plausibility[f]:.0%}' if plausibility.get(f) is not None else '—'
    soft = f'{soft_sim[f]:.2f}'
    strict = f'{strict_acc(f):.0%}'
    print(f'{f:<26} {cov:>10} {plaus:>12} {soft:>10} {strict:>9}')

# Сводные средние
avg_cov = sum(coverage.values()) / len(coverage)
avg_soft = sum(soft_sim.values()) / len(soft_sim)
print('─' * 70)
print(f'{"AVERAGE":<26} {avg_cov:>9.0%}{"":>12} {avg_soft:>10.2f} {sum(strict_acc(f) for f in EVAL_FIELDS)/len(EVAL_FIELDS):>9.0%}')

# Сохраняем в eval_report
report = json.loads(EVAL_REPORT.read_text()) if EVAL_REPORT.exists() else {}
report['coverage']     = coverage
report['plausibility'] = plausibility
report['soft_sim']     = soft_sim
EVAL_REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=1))
print(f'\nЗаписан в {EVAL_REPORT}')

FIELD                        COVERAGE    PLAUSIBLE   SOFT SIM    STRICT
──────────────────────────────────────────────────────────────────────
product_name                      38%            —       0.40        0%
price_default                     28%         100%       0.25        1%
price_card                        58%         100%       0.70       62%
price_discount                     0%            —       1.00      100%
barcode                            7%          61%       0.12        4%
discount_amount                   32%         100%       0.59       59%
id_sku                             4%          82%       0.12        6%
print_datetime                     0%            —       0.34       34%
code                               0%            —       0.25       25%
additional_info                    3%         100%       0.57       50%
color                            100%         100%       0.99       99%
special_symbols                    0%            —       0.22    